# Phoneme/Diacritics and Built (Rule-Based) Lab

This notebook demonstrates a minimal, deterministic pipeline: normalization + affix-to-base, syllabification (six gates), rule-based detectors for pronouns/particles/built verbs, and summary stats/plots.

In [ ]:
# Deterministic UTF-8 separation helpers
from src.utf8_stream import (
    classify_char, extract_streams, filter_to_phoneme_text, drop_non_phoneme, summarize_ignored
)

sample = "مُحَمَّدٌ رَسُولُ ٱللَّهِ [pause] ، وَهُوَ الرَّحْمٰنُ الرَّحِيمُ"
stream, ignored = extract_streams(sample)
len(stream), len(ignored)

In [ ]:
# Inspect the first few phoneme entries (C + optional F/K/D/S)
for pc in stream[:12]:
    print(f"{pc.raw!r} -> (ch={pc.ch}, dia={pc.dia})")

In [ ]:
# Show ignored characters summary with codepoints and counts
rows = summarize_ignored(ignored)
# If pandas is available, make a small DataFrame for nicer view
try:
    import pandas as pd
    import numpy as np
    df_ign = pd.DataFrame(rows)
    # Rename 'class_' to 'class' for display
    if 'class_' in df_ign.columns:
        df_ign = df_ign.rename(columns={'class_': 'class'})
    display(df_ign.head(20))
except Exception:
    for r in rows[:20]:
        cp = r['codepoint']
        print(f"{r['char']!r} ({cp}) -> {r['class_']} x{r['count']}")

In [ ]:
# Filter the text to the pure phoneme stream (letters + attached marks)
filtered = filter_to_phoneme_text(sample)
print(filtered)

In [ ]:
# Use on a larger text if available
try:
    with open('../data/quran-simple-enhanced.txt', 'r', encoding='utf-8') as f:
        big_text = f.read()[:5000]  # sample for speed
    _, ig2 = extract_streams(big_text)
    rows2 = summarize_ignored(ig2)
    try:
        import pandas as pd
        display(pd.DataFrame(rows2).head(20))
    except Exception:
        print(rows2[:5])
except FileNotFoundError:
    print("quran-simple-enhanced.txt not found; skipped.")

In [ ]:
# Strict vs Lenient comparison on a sample
from src.utf8_stream import extract_streams, summarize_ignored, save_ignored_csv

sample = "مُحَمَّدٌ رَسُولُ ٱللَّهِ [pause] ، وَهُوَ الرَّحْمٰنُ الرَّحِيمُ"
_, ignored_len = extract_streams(sample, strict_aux=False)
_, ignored_str = extract_streams(sample, strict_aux=True)
rows_len = summarize_ignored(ignored_len)
rows_str = summarize_ignored(ignored_str)

try:
    import pandas as pd
    print("Lenient (keeps AUX in raw):")
    display(pd.DataFrame(rows_len).head(10))
    print("\nStrict (AUX always ignored):")
    display(pd.DataFrame(rows_str).head(10))
except Exception:
    print(rows_len[:5])
    print(rows_str[:5])

# Optionally save CSVs
# save_ignored_csv('../reports/ignored_lenient.csv', rows_len)
# save_ignored_csv('../reports/ignored_strict.csv', rows_str)